In [43]:
#utils
import pandas as pd
import numpy as np
import contextlib
import os
import warnings
from scipy.stats import norm

#visualization libraries
import matplotlib.pyplot as plt

#machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve

#models 
from models.AutoEncoder import Autoencoder
from models.ContractiveAutoEncoder import ContractiveAutoEncoder
from models.VariationalAutoencoder import VariationalAutoEncoder
from models.alad.ALAD import ALAD


warnings.filterwarnings("ignore")

In [44]:
df = pd.read_csv('SimulationData.csv')
df = df.head(200)

In [45]:
target_feature = "isghost"
columns = df.drop(columns=target_feature).columns
#instantiate two more datasets one to contain the normal data
#and the other to hold the anomalous instances
train_normal_data = df[df[target_feature] == 0].drop(columns=target_feature)
anomalous_data = df[df[target_feature] == 1].drop(columns=target_feature)

In [46]:
def get_normal_data(df):
    df = df.sample(frac=1)
    features = df.drop([target_feature], axis=1)
    label = df[target_feature]

    #the stratify parameter here is set to be "isghost" meaning that the 
    #proportions of ghost and normal tracks in both train and test datasets
    #will be kept the same...
    x_train, x_test, y_train, y_test = train_test_split(
        features, label, test_size=0.2, stratify=label
    )

    #remember that at least for the autoencoders we're gonna use just the 
    #normal instances during the training phase so we have to filter those in x_train
    train_index = y_train[y_train == 0].index
    train_normal_data = x_train.loc[train_index]

    return train_normal_data, x_train, y_train, x_test, y_test

In [47]:
train_normal_data, x_train, y_train, x_test, y_test = get_normal_data(df)

In [48]:
def standardize_normalize(train_x, normal_train_data, test_data):

    #creating a scaler and decing which columns to process
    normal_train_data = (normal_train_data - normal_train_data.mean()) / normal_train_data.std()
    train_x  =  (train_x - train_x.mean()) / train_x.std()
    test_data = (test_data - normal_train_data.mean()) / normal_train_data.std()

    #now moving the data to a fixed range of values 
    scaler = MinMaxScaler()

    normal_train_data = scaler.fit_transform(normal_train_data)
    train_x = scaler.fit_transform(train_x)
    test_data = scaler.fit_transform(test_data)

    return train_x, normal_train_data, test_data

In [49]:
x_train, train_normal_data, x_test = standardize_normalize(x_train, train_normal_data, x_test)

In [50]:
INPUT_DIM = train_normal_data.shape[1]
ENCODING_DIM = 3
learing_rates = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05]

In [51]:
def get_data():

    train_normal_data, _, _, x_test, y_test = get_normal_data(df)
    _, train_normal_data, x_test = standardize_normalize(x_train, train_normal_data, x_test)

    return train_normal_data, x_test, y_test

In [52]:
def build_models(learning_rate, encoding_dim):

    ae = Autoencoder(INPUT_DIM, encoding_dim, learning_rate)
    cae = ContractiveAutoEncoder(INPUT_DIM, encoding_dim, lambda_=0.01, learning_rate = learning_rate)
    vae = VariationalAutoEncoder(INPUT_DIM, encoding_dim, learning_rate)
    alad = ALAD(input_dim=INPUT_DIM, latent_dim=encoding_dim, learning_rate=learning_rate)

    ae.build(input_shape=(train_normal_data.shape))
    cae.build(input_shape=(train_normal_data.shape))
    vae.build(input_shape=(train_normal_data.shape))
    
    return ae, cae, vae, alad

In [53]:

def avgNestedLists(nested_vals):
    """
    Averages a 2-D array and returns a 1-D array of all of the columns
    averaged together, regardless of their dimensions.
    """
    output = []
    maximum = 0
    for lst in nested_vals:
        if len(lst) > maximum:
            maximum = len(lst)
    for index in range(maximum): # Go through each index of longest list
        temp = []
        for lst in nested_vals: # Go through each list
            if index < len(lst): # If not an index error
                temp.append(lst[index])
        output.append(np.nanmean(temp))
    return output

In [54]:
import numpy as np
from sklearn.metrics import roc_curve, auc
from scipy import stats


def mean_and_confidence(model, threshold_function, alpha=0.95, iterations=5, transfer=False, epochs=5, learning_rate = 0.003):
   

    #initialize all the containers for teh metrics to track
    list_fpr = []
    list_tpr = []
    list_auc = []

    #precision and recall will be used for 
    #the curve but also as general parameters 
    list_precision_curve = []
    list_recall_curve = []

    #classification metrics 
    list_accuracy = []
    list_precision = []
    list_recall = []
    list_f1 = []

    list_training_time = []

    model.save_weights('saved_weights/my_model' +  model.model_name)

    

    # Perform 100 iterations
    for _ in range(iterations):

        x_train, x_test, y_test = get_data()

        history, training_time = model.train(x_train, epochs=epochs, batch_size=30, validation_split=0.2)
        ###time each model 
        threshold = threshold_function(x_train, x_test, y_test)
        predictions = model.find_anomalies(x_test, threshold)
        accuracy, precision, recall, f1 = model.classification_metrics(y_test, predictions)

        #re-set all the trainable variables of the model
        model.load_weights('saved_weights/my_model' +  model.model_name)
        
        fpr_ae, tpr_ae, _ = roc_curve(y_test, predictions)
        auc_ae = auc(fpr_ae, tpr_ae)
        list_fpr.append(fpr_ae)
        list_tpr.append(tpr_ae)
        

        precision_curve, recall_curve, _ = precision_recall_curve(y_test, predictions)
        list_precision_curve.append(precision_curve)
        list_recall_curve.append(recall_curve)


        list_accuracy.append(accuracy)
        list_precision.append(precision)
        list_recall.append(recall)
        list_f1.append(f1)
        list_training_time.append(training_time)
        list_auc.append(auc_ae)

    #roc_curve can return different sized arrays
    #this means that ther needs to be a custom function
    #to compute the mean 
    avg_fpr = avgNestedLists(list_fpr)
    avg_tpr = avgNestedLists(list_tpr)
    avg_precision_curve = avgNestedLists(list_precision_curve)
    avg_recall_curve = avgNestedLists(list_recall_curve)
    avg_acc = np.mean(list_accuracy)
    avg_rec = np.mean(list_recall)
    avg_pre = np.mean(list_precision)
    avg_f1 = np.mean(list_f1)
    avg_time = np.mean(list_training_time)
    avg_auc = np.mean(list_auc)


    #standard deviations
    accuracy_std = np.std(list_accuracy)
    precision_std = np.std(list_precision)
    recall_std = np.std(list_recall)
    f1_std = np.std(list_f1)
    training_time_std = np.std(list_training_time)
    auc_std = np.std(list_auc)

    #how to take the "mean curve"  
    interval_acc = stats.t.interval(alpha=alpha, loc=np.mean(list_accuracy), df=len(list_accuracy)-1, scale=stats.sem(list_accuracy))
    interval_rec = stats.t.interval(alpha=alpha, loc=np.mean(list_recall), df=len(list_recall)-1, scale=stats.sem(list_recall))
    interval_pre = stats.t.interval(alpha=alpha, loc=np.mean(list_precision), df=len(list_precision)-1, scale=stats.sem(list_precision))
    interval_f1 = stats.t.interval(alpha=alpha, loc=np.mean(list_f1), df=len(list_f1)-1, scale=stats.sem(list_f1))
    interval_time = stats.t.interval(alpha=alpha, loc=np.mean(list_training_time), df=len(list_training_time)-1, scale=stats.sem(list_training_time))
    interval_auc = stats.t.interval(alpha=alpha, loc=np.mean(list_auc), df=len(list_auc)-1, scale=stats.sem(list_auc))

    plt.plot(avg_fpr, avg_tpr)
    plt.plot(avg_fpr, avg_tpr, color='darkorange', lw=2, label='ROC curve (AUC = %0.2f)' % avg_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve ' + model.model_name + " ")
    plt.legend(loc="lower right")
    plt.savefig('images/Average ROC' + model.model_name)
    plt.close()

    plt.plot(avg_precision_curve, avg_recall_curve)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve ' + model.model_name + " ")
    plt.savefig('images/Average Precision Recall ' + model.model_name)
    plt.close()


    averages = {
        "Average FPR": avg_fpr,
        "Average TPR": avg_tpr,
        "Average precision curve": avg_precision_curve,
        "Average recall curve": avg_recall_curve,
        "Average accuracy": avg_acc,
        "Average recall": avg_rec,
        "Average precision": avg_pre,
        "Average F-1": avg_f1,
        "Average training time": avg_time,
        "Average AUC": avg_auc,
        }
    

    sigmas = {
        "STD accuracy": accuracy_std,
        "STD recall": recall_std,
        "STD precision": precision_std,
        "STD F-1": f1_std,
        "STD training time": training_time_std,
        "STD AUC": auc_std,
    }

    
    intervals = {
        "Interval accuracy": interval_acc,
        "Interval recall": interval_rec,
        "Interval precision": interval_pre,
        "Interval F-1": interval_f1,
        "Interval training time": interval_time,
        "Interval AUC": interval_auc,
        }
    
    return [averages, intervals, sigmas]

# Experiment with learning rate

In [58]:
with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    alad_list = []
    ae_list = []
    cae_list = []
    vae_list = []

    for rate in learing_rates:

        ae, cae, vae, alad = build_models(encoding_dim=3, learning_rate=rate)
        
        alad_list.append(mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)[0])
        print("ALAD")
        ae_list.append(mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)[0])
        print("AE")
        cae_list.append(mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)[0])
        print("CAE")
        vae_list.append(mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)[0])


### I'm using the recall as a parameter ot be optimized for the learning_rate search.

In [ ]:
def find_best(method_list, variable_list):

    maxx = 0

    for index, value in enumerate(method_list):

        if value["Average recall"] > maxx:
            maxx = value["Average recall"]
            best_rate = variable_list[index]

    return str(best_rate)

print("the best rate for alad method is " + find_best(alad_list, learing_rates))
print("the best rate for ae method is " + find_best(ae_list, learing_rates))
print("the best rate for cae method is " + find_best(cae_list, learing_rates))
print("the best rate for vae method is " + find_best(vae_list, learing_rates))

In [57]:
best_rate = 0.003

# Experiment with encoding dimension

In [56]:
encoding_dims = list(range(2, round(INPUT_DIM / 2),  1))
encoding_dims = [2, 3, 5]

with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    alad_list = []
    ae_list = []
    cae_list = []
    vae_list = []

    for encoding_dim in encoding_dims:

        ae, cae, vae, alad = build_models(encoding_dim=encoding_dim, learning_rate=best_rate)
        
        alad_list.append(mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95))
        print("ALAD")
        ae_list.append(mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95))
        print("AE")
        cae_list.append(mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95))
        print("CAE")
        vae_list.append(mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95))

NameError: name 'best_rate' is not defined

In [ ]:
print("the best encoding_dim for alad method is " + find_best(alad_list, learing_rates))
print("the best encoding_dim for ae method is " + find_best(ae_list, learing_rates))
print("the best encoding_dim for cae method is " + find_best(cae_list, learing_rates))
print("the best encoding_dim for vae method is " + find_best(vae_list, learing_rates))